# BanglaGuard-Base — Full Benchmark + Full LODO Cross-Domain Experiment

This notebook combines the supplied **BanglaGuard XLM-R Base full experiment**
and the supplied **cross-domain LODO experiment** into one coherent Colab pipeline.

## Stage A — Full BanglaGuard-Base benchmark

Uses the normal `train / val / heldout_test` protocol and produces:

- multi-seed training;
- optional weight soup;
- temperature calibration;
- class-conditional conformal analysis;
- decision-matched diagnostic;
- per-language-form analysis;
- **domain-wise analysis**;
- intent-wise analysis;
- hard-negative over-refusal;
- threat recall / AUROC / PR-AUC;
- calibration / ECE / Brier;
- unseen robustness stress;
- inference-efficiency report.

## Stage B — Full LODO (Leave-One-Domain-Out)

Runs three independent domain-generalization experiments:

| LODO target | Training + validation domains | Unseen test domain |
|---|---|---|
| Banking | Healthcare + E-commerce | Banking |
| Healthcare | Banking + E-commerce | Healthcare |
| E-commerce | Banking + Healthcare | E-commerce |

For every LODO fold, the unseen target domain is excluded from:

- supervised training;
- validation and early stopping;
- TAPT/TLM adaptation;
- rationale-alignment supervision.

At unseen-target inference, the model uses the **generic policy embedding**.

## Final domain-wise outputs

The notebook produces a direct comparison for each domain:

- standard in-domain heldout macro-F1;
- unseen LODO macro-F1;
- generalization gap;
- REVIEW F1;
- threat recall;
- hard-negative over-refusal;
- decision-matched performance;
- per-form LODO performance.

### Colab

1. **Runtime → Change runtime type → GPU**
2. Run Cell 1.
3. Upload `splits_v2.zip` in Cell 2.
4. Run all remaining cells.

The notebook is configured for a **full experiment by default**.

In [ ]:
# ============================================================
# CELL 1 — INSTALL + FULL EXPERIMENT CONFIG
# ============================================================
!pip -q install -U "transformers>=4.50,<5" "accelerate>=1.0" \
    "sentencepiece>=0.1.99" "safetensors>=0.4" \
    "scikit-learn>=1.3" "datasets>=2.18"

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re, gc, json, math, time, random, shutil, zipfile, warnings, unicodedata
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, matthews_corrcoef,
    roc_auc_score, average_precision_score
)

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling, get_linear_schedule_with_warmup
)

from google.colab import files as colab_files

warnings.filterwarnings("ignore")

# ---------------- Model ----------------
BACKBONE = "FacebookAI/xlm-roberta-base"
MODEL_SLUG = "banglaguard_base_full_plus_lodo"

# ---------------- Experiment switches ----------------
# Full experiment requested:
SMOKE_RUN = False
RUN_MAIN_EXPERIMENT = True
RUN_FULL_LODO = True

# Main benchmark keeps the original five-seed protocol.
MAIN_SEEDS = [42, 43, 44, 45, 46]

# LODO notebook used three seeds per unseen target domain.
LODO_SEEDS = [42, 43, 44]

FIXED_HEAD_INIT_SEED = 2026

USE_TAPT_TLM = True
USE_RATIONALE_ALIGNMENT = True
USE_WEIGHT_SOUP = True
USE_CONFORMAL = True
RUN_ROBUSTNESS_STRESS = True

# No genuine OOF teacher artifact was supplied.
# KD is therefore deliberately disabled.
USE_KD = False

# Saving the model is not required for paper metrics.
# Change to True if you also need the deployment checkpoint.
SAVE_FINAL_MODEL = False

# ---------------- Task ----------------
TEXT_VIEWS = ["bangla", "banglish", "code_mix", "english"]

LABELS = ["ALLOW", "REVIEW", "BLOCK"]
LABEL2ID = {x:i for i,x in enumerate(LABELS)}
ID2LABEL = {i:x for x,i in LABEL2ID.items()}

DOMAINS = ["banking", "healthcare", "e-commerce"]
DOMAIN2ID = {x:i for i,x in enumerate(DOMAINS)}

INTENTS = [
    "B0_legitimate_security",
    "S1_jailbreak",
    "S2_prompt_extraction",
    "S3_data_exfiltration",
    "S4_unauthorized_action",
    "S5_authentication_authorization_bypass",
    "S6_indirect_injection",
]
INTENT2ID = {x:i for i,x in enumerate(INTENTS)}

RISKS = ["none", "low", "medium", "high", "very_high"]
RISK2ID = {x:i for i,x in enumerate(RISKS)}

# ---------------- Hyperparameters ----------------
MAX_LEN = 128
EPOCHS = 12
PATIENCE = 3
EARLY_STOP_MIN_DELTA = 1e-3

LR_ENCODER = 3e-5
LR_HEADS = 3e-4
LAYERWISE_LR_DECAY = 0.90
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

DROPOUT = 0.10
POLICY_DROPOUT = 0.20
LABEL_SMOOTHING = 0.05
AUGMENT_PROB = 0.30

LAMBDA_INT = 0.50
LAMBDA_DOM = 0.10
LAMBDA_ORD = 0.30
LAMBDA_CONS = 1.00
LAMBDA_SCL = 0.10
LAMBDA_RAT = 0.20
SUPCON_TAU = 0.07

CLASS_W = torch.tensor([0.96, 1.30, 0.74], dtype=torch.float32)

# Effective batch = 16 scenarios × 4 views = 64 texts.
# Conservative settings for Colab T4.
TRAIN_SCENARIO_BATCH = 2
GRAD_ACCUM = 8
EVAL_SCENARIO_BATCH = 4

# TAPT/TLM
TAPT_EPOCHS = 3
TAPT_LR = 3e-5
TAPT_BATCH = 2
TAPT_GRAD_ACCUM = 4

RATIONALE_MODEL = "intfloat/multilingual-e5-large"

# Smoke values remain available if you manually switch SMOKE_RUN=True.
SMOKE_TRAIN_N = 48
SMOKE_VAL_N = 24
SMOKE_TEST_N = 36
SMOKE_MATCHED_N = 24
SMOKE_MAX_STEPS = 6
SMOKE_TAPT_STEPS = 4

N_BOOT = 200 if SMOKE_RUN else 10000

# ---------------- Runtime ----------------
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
CC = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)

AMP_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and CC[0] >= 8
    else torch.float16
)
USE_SCALER = torch.cuda.is_available() and AMP_DTYPE == torch.float16

WORK_ROOT = Path("/content")
UPLOAD_ROOT = WORK_ROOT / "runtime_uploads"
UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)

OUT_ROOT = WORK_ROOT / MODEL_SLUG
MAIN_DIR = OUT_ROOT / "main_benchmark"
LODO_DIR = OUT_ROOT / "lodo"
CACHE_DIR = OUT_ROOT / "cache"

for p in [OUT_ROOT, MAIN_DIR, LODO_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        gc.collect()

def amp_context():
    if torch.cuda.is_available():
        return torch.autocast("cuda", dtype=AMP_DTYPE)
    return nullcontext()

print("GPU:", GPU_NAME)
print("AMP dtype:", AMP_DTYPE)
print("SMOKE_RUN:", SMOKE_RUN)
print("MAIN seeds:", [42] if SMOKE_RUN else MAIN_SEEDS)
print("LODO seeds:", [42] if SMOKE_RUN else LODO_SEEDS)
print("RUN_MAIN_EXPERIMENT:", RUN_MAIN_EXPERIMENT)
print("RUN_FULL_LODO:", RUN_FULL_LODO)
print("OUT_ROOT:", OUT_ROOT)

if not torch.cuda.is_available():
    print("\nWARNING: Enable a GPU from Runtime → Change runtime type → GPU.")

In [ ]:
# ============================================================
# CELL 2 — UPLOAD splits_v2 INTO COLAB RUNTIME
# ============================================================
print("Upload splits_v2.zip (recommended).")
print("A direct SecBangla_clean_master_with_splits.csv also works.\n")

uploaded = colab_files.upload()

if not uploaded:
    raise RuntimeError("No dataset was uploaded.")

for name, payload in uploaded.items():
    dst = UPLOAD_ROOT / Path(name).name
    with open(dst, "wb") as f:
        f.write(payload)
    print(f"saved: {dst} ({len(payload)/1024**2:.2f} MB)")

In [ ]:
# ============================================================
# CELL 3 — LOAD MASTER DATASET + AUDIT + MAIN SPLITS
# ============================================================
MASTER_FILENAME = "SecBangla_clean_master_with_splits.csv"
EXTRACT_ROOT = WORK_ROOT / "_splits_v2_extract"
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

def find_master(root):
    hits = list(Path(root).rglob(MASTER_FILENAME))
    return hits[0] if hits else None

master_path = find_master(UPLOAD_ROOT)

if master_path is None:
    zips = list(UPLOAD_ROOT.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(
            f"Upload splits_v2.zip or {MASTER_FILENAME}."
        )

    for z in zips:
        print("extracting:", z.name)
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(EXTRACT_ROOT)
        master_path = find_master(EXTRACT_ROOT)
        if master_path is not None:
            break

if master_path is None:
    raise FileNotFoundError(
        f"{MASTER_FILENAME} was not found in the uploaded archive."
    )

MASTER_FULL = pd.read_csv(master_path)

REQUIRED = [
    "id", "domain", "security_intent", "hard_negative_type",
    "decision", "risk_level", "note",
    "bangla", "banglish", "code_mix", "english",
    "scenario_family_id", "split", "in_decision_matched_test",
]

missing = [c for c in REQUIRED if c not in MASTER_FULL.columns]
assert not missing, f"Missing columns: {missing}"
assert MASTER_FULL.id.is_unique
assert set(MASTER_FULL.domain) == set(DOMAINS)
assert set(MASTER_FULL.decision) <= set(LABELS)

for v in TEXT_VIEWS:
    assert MASTER_FULL[v].notna().all(), f"Null text found in {v}"

def to_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().isin(
        ["true", "1", "yes", "y"]
    )

MASTER_FULL["in_decision_matched_test"] = to_bool(
    MASTER_FULL["in_decision_matched_test"]
)

MAIN_TRAIN_FULL = MASTER_FULL[
    MASTER_FULL.split == "train"
].copy().reset_index(drop=True)

MAIN_VAL_FULL = MASTER_FULL[
    MASTER_FULL.split == "val"
].copy().reset_index(drop=True)

MAIN_TEST_FULL = MASTER_FULL[
    MASTER_FULL.split == "heldout_test"
].copy().reset_index(drop=True)

MAIN_MATCHED_FULL = MAIN_TEST_FULL[
    MAIN_TEST_FULL.in_decision_matched_test
].copy().reset_index(drop=True)

# Family leakage audit.
assert set(MAIN_TRAIN_FULL.scenario_family_id).isdisjoint(
    set(MAIN_VAL_FULL.scenario_family_id)
)
assert set(MAIN_TRAIN_FULL.scenario_family_id).isdisjoint(
    set(MAIN_TEST_FULL.scenario_family_id)
)
assert set(MAIN_VAL_FULL.scenario_family_id).isdisjoint(
    set(MAIN_TEST_FULL.scenario_family_id)
)

for v in TEXT_VIEWS:
    tr = set(MAIN_TRAIN_FULL[v].astype(str))
    assert tr.isdisjoint(set(MAIN_VAL_FULL[v].astype(str)))
    assert tr.isdisjoint(set(MAIN_TEST_FULL[v].astype(str)))

def smoke_sample(df, n, seed):
    if len(df) <= n:
        return df.copy().reset_index(drop=True)

    tmp = df.copy()
    tmp["_stratum"] = (
        tmp.domain.astype(str) + "|" + tmp.decision.astype(str)
    )
    groups = list(tmp.groupby("_stratum", sort=False))
    rng = np.random.RandomState(seed)
    per = max(1, n // max(len(groups),1))
    parts = []

    for _,g in groups:
        parts.append(
            g.sample(
                min(per, len(g)),
                random_state=int(rng.randint(2**31-1))
            )
        )

    out = pd.concat(parts).drop(
        columns="_stratum"
    ).drop_duplicates("id")

    if len(out) < n:
        rem = df[~df.id.isin(out.id)]
        if len(rem):
            out = pd.concat([
                out,
                rem.sample(
                    min(n-len(out), len(rem)),
                    random_state=seed
                )
            ])

    return out.sample(
        min(n,len(out)),
        random_state=seed
    ).reset_index(drop=True)

if SMOKE_RUN:
    MAIN_TRAIN = smoke_sample(MAIN_TRAIN_FULL, SMOKE_TRAIN_N, 42)
    MAIN_VAL = smoke_sample(MAIN_VAL_FULL, SMOKE_VAL_N, 43)
    MAIN_TEST = smoke_sample(MAIN_TEST_FULL, SMOKE_TEST_N, 44)
    MAIN_MATCHED = MAIN_TEST[
        MAIN_TEST.in_decision_matched_test
    ].copy().reset_index(drop=True)
else:
    MAIN_TRAIN = MAIN_TRAIN_FULL.copy()
    MAIN_VAL = MAIN_VAL_FULL.copy()
    MAIN_TEST = MAIN_TEST_FULL.copy()
    MAIN_MATCHED = MAIN_MATCHED_FULL.copy()

print("Master:", master_path)
print(
    "counts:",
    {
        "train":len(MAIN_TRAIN),
        "val":len(MAIN_VAL),
        "heldout":len(MAIN_TEST),
        "decision_matched":len(MAIN_MATCHED),
    }
)
display(pd.crosstab(
    MAIN_TEST.domain,
    MAIN_TEST.decision,
    margins=True
))
print("Main leakage audit: PASS")

In [ ]:
# ============================================================
# CELL 3 — SCRIPT-AWARE NORMALIZER + B1–B4 AUGMENTATION
# ============================================================
_ZW = re.compile("[\u200b\u2060\ufeff]")  # keep U+200C/U+200D
_HOMO = str.maketrans({
    "а":"a","е":"e","о":"o","р":"p","с":"c","х":"x","і":"i","ѕ":"s",
    "ο":"o","α":"a","ε":"e","ρ":"p"
})
_ASCII_TO_HOMO={"a":"а","e":"е","o":"о","p":"р","c":"с","x":"х","i":"і","s":"ѕ"}

def normalize_security_text(text):
    t=unicodedata.normalize("NFKC",str(text))
    t=_ZW.sub("",t).translate(_HOMO)
    frag=re.compile(r"(?i)\b(?:[a-z][\s\-._*]){3,}[a-z]\b")
    t=frag.sub(lambda m:re.sub(r"[\s\-._*]","",m.group(0)),t)
    return re.sub(r"\s+"," ",t).strip()

# Approximate ~60-rule Banglish orthographic alternation table.
BANGLISH_RULES=[
    ("sh","s"),("s","sh"),("chh","ch"),("ch","chh"),("bh","v"),("v","bh"),
    ("ph","f"),("f","ph"),("kh","k"),("gh","g"),("th","t"),("dh","d"),
    ("aa","a"),("ee","i"),("ii","i"),("oo","u"),("ou","o"),("au","o"),
    ("achhe","ache"),("ache","ase"),("ase","ache"),("nai","nei"),("nei","nai"),
    ("koro","kro"),("kro","koro"),("koru","koro"),("hobe","hbe"),("hbe","hobe"),
    ("amar","amr"),("tomar","tmr"),("apnar","apnr"),("account","acount"),
    ("password","passwrd"),("verify","verfy"),("verification","verfication"),
    ("please","pls"),("plz","pls"),("message","msg"),("number","num"),
    ("transaction","transction"),("payment","paymnt"),("transfer","transfr"),
    ("security","securty"),("system","systm"),("prompt","promt"),("instruction","instrction"),
    ("kivabe","kibhabe"),("kibhabe","kivabe"),("ki","kii"),("ami","amii"),
    ("taka","tk"),("ekhon","ekhn"),("lagbe","lgbe"),("dorkar","drkar"),
    ("korte","krte"),("korbo","krbo"),("bolo","blo"),("dao","daw"),
    ("diye","dia"),("jabe","jbe"),("parbo","prbo"),("chai","chy"),
]

LOANWORD_FLIPS={
    "debit card":"ডেবিট কার্ড","credit card":"ক্রেডিট কার্ড","account":"অ্যাকাউন্ট",
    "otp":"ওটিপি","password":"পাসওয়ার্ড","refund":"রিফান্ড","invoice":"ইনভয়েস",
    "payment":"পেমেন্ট","transaction":"ট্রানজ্যাকশন","transfer":"ট্রান্সফার",
    "security":"সিকিউরিটি","system":"সিস্টেম","prompt":"প্রম্পট","login":"লগইন",
    "verification":"ভেরিফিকেশন","email":"ইমেইল","sms":"এসএমএস","api":"এপিআই",
}
CARRIERS=[
    "SMS content: {x}","Email body: {x}","Invoice note: {x}","Résumé text: {x}",
    "JSON field `message`: {x}","Web snippet: {x}","Vendor memo: {x}"
]

def _obfuscate(t,rng):
    toks=t.split()
    if not toks: return t
    mode=rng.choice(["fragment","spacing","zero_width","homoglyph","punct","repeat","hasanta_drop"])
    j=rng.randrange(len(toks)); tok=toks[j]
    if mode=="fragment" and tok.isascii() and tok.isalpha() and len(tok)>=5:
        toks[j]="-".join(tok)
    elif mode=="spacing" and tok.isascii() and tok.isalpha() and len(tok)>=5:
        toks[j]=" ".join(tok)
    elif mode=="zero_width" and len(tok)>=4:
        k=max(1,len(tok)//2); toks[j]=tok[:k]+"\u200b"+tok[k:]
    elif mode=="homoglyph" and tok.isascii():
        chars=list(tok)
        cand=[i for i,c in enumerate(chars) if c.lower() in _ASCII_TO_HOMO]
        if cand:
            k=rng.choice(cand); chars[k]=_ASCII_TO_HOMO[chars[k].lower()]
            toks[j]="".join(chars)
    elif mode=="punct" and len(tok)>=4:
        toks[j]=tok+"..."
    elif mode=="repeat" and len(tok)>=3:
        k=rng.randrange(len(tok)); toks[j]=tok[:k]+tok[k]*2+tok[k:]
    elif mode=="hasanta_drop" and "্" in tok:
        toks[j]=tok.replace("্","",1)
    return " ".join(toks)

def augment_text(text,view,intent,rng):
    original=str(text); t=original

    # B1 Banglish orthography
    if view in ("banglish","code_mix"):
        for _ in range(rng.randint(1,3)):
            a,b=rng.choice(BANGLISH_RULES)
            t=re.sub(re.escape(a),b,t,count=1,flags=re.I)

    # B2 loanword script flip (20–50%)
    if view in ("bangla","code_mix"):
        flip_p=rng.uniform(0.20,0.50)
        for a,b in LOANWORD_FLIPS.items():
            if rng.random()<flip_p:
                t=re.sub(re.escape(a),b,t,flags=re.I)

    # B3 L5 obfuscation, all classes
    t=_obfuscate(t,rng)

    # B4 carrier wrapping for S6 and benign controls
    if intent in ("S6_indirect_injection","B0_legitimate_security") and rng.random()<0.50:
        t=rng.choice(CARRIERS).format(x=t)

    if t==original:
        t=original+" ."
    return t

print(normalize_security_text("s-y-s-t-e-m   prompt দেখাও"))

In [ ]:
# ============================================================
# CELL 5 — TOKENIZER + LENGTH AUDIT
# ============================================================
TOKENIZER = AutoTokenizer.from_pretrained(
    BACKBONE,
    use_fast=True,
)

def truncation_audit(df):
    counts = []
    for v in TEXT_VIEWS:
        enc = TOKENIZER(
            df[v].astype(str).tolist(),
            add_special_tokens=True,
            truncation=False,
        )
        counts.extend(len(x) for x in enc["input_ids"])

    arr = np.asarray(counts)
    return {
        "p50":float(np.percentile(arr,50)),
        "p95":float(np.percentile(arr,95)),
        "p99":float(np.percentile(arr,99)),
        "max":int(arr.max()),
        "fraction_over_max_len":float((arr > MAX_LEN).mean()),
    }

length_stats = truncation_audit(MAIN_TRAIN_FULL)
print("token-length audit:", length_stats)

if length_stats["fraction_over_max_len"] > 0.005:
    print(
        "WARNING: >0.5% of views exceed MAX_LEN=128. "
        "Consider MAX_LEN=192 in a separate controlled ablation."
    )

In [ ]:
# ============================================================
# CELL 6 — PARAMETERIZED TAPT/TLM
# ============================================================
class MLMTextDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return TOKENIZER(
            self.texts[i],
            truncation=True,
            max_length=MAX_LEN,
            return_special_tokens_mask=True,
        )

def build_tapt_tlm_texts(df):
    sep = TOKENIZER.sep_token or "</s>"
    texts = []

    for _,r in df.iterrows():
        views = [
            normalize_security_text(r[v])
            for v in TEXT_VIEWS
        ]

        texts.extend(views)
        texts.append(views[0] + f" {sep} " + views[1])
        texts.append(views[1] + f" {sep} " + views[3])
        texts.append(views[2] + f" {sep} " + views[0])

    return texts

def prepare_tapt_backbone(train_df, tag):
    """
    TAPT/TLM is trained only on the supplied training dataframe.
    For LODO, train_df contains source domains only.
    """
    if not USE_TAPT_TLM:
        return BACKBONE

    out_dir = CACHE_DIR / "tapt" / tag

    if (out_dir / "config.json").exists():
        print("Using cached TAPT/TLM:", out_dir)
        return str(out_dir)

    print(f"TAPT/TLM [{tag}] | scenarios={len(train_df)}")
    out_dir.mkdir(parents=True, exist_ok=True)

    # Keep all CUDA-heavy locals inside this function.
    mlm = AutoModelForMaskedLM.from_pretrained(
        BACKBONE
    ).to(DEVICE)

    if hasattr(mlm, "gradient_checkpointing_enable"):
        mlm.gradient_checkpointing_enable()

    texts = build_tapt_tlm_texts(train_df)
    if SMOKE_RUN:
        texts = texts[:192]

    ds = MLMTextDataset(texts)
    collator = DataCollatorForLanguageModeling(
        tokenizer=TOKENIZER,
        mlm_probability=0.15,
    )

    gen = torch.Generator()
    gen.manual_seed(2026)

    dl = DataLoader(
        ds,
        batch_size=TAPT_BATCH,
        shuffle=True,
        collate_fn=collator,
        generator=gen,
    )

    opt = torch.optim.AdamW(
        mlm.parameters(),
        lr=TAPT_LR,
        weight_decay=WEIGHT_DECAY,
        foreach=False,
    )
    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_SCALER
    )

    epochs = 1 if SMOKE_RUN else TAPT_EPOCHS
    max_updates = SMOKE_TAPT_STEPS if SMOKE_RUN else None
    updates_per_epoch = max(
        1,
        math.ceil(len(dl)/TAPT_GRAD_ACCUM)
    )
    total_updates = (
        max_updates
        if max_updates
        else updates_per_epoch*epochs
    )

    sched = get_linear_schedule_with_warmup(
        opt,
        int(WARMUP_RATIO*total_updates),
        total_updates,
    )

    update = 0
    opt.zero_grad(set_to_none=True)
    stop = False

    for ep in range(epochs):
        mlm.train()

        for bi,batch in enumerate(dl):
            batch = {
                k:v.to(DEVICE)
                for k,v in batch.items()
            }

            with amp_context():
                loss = (
                    mlm(**batch).loss
                    / TAPT_GRAD_ACCUM
                )

            scaler.scale(loss).backward()

            if (
                (bi+1)%TAPT_GRAD_ACCUM == 0
                or (bi+1) == len(dl)
            ):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    mlm.parameters(),
                    MAX_GRAD_NORM,
                )
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
                update += 1

                if (
                    update%20 == 0
                    or (
                        max_updates
                        and update >= max_updates
                    )
                ):
                    print(
                        "  TAPT",
                        {
                            "epoch":ep+1,
                            "update":update,
                            "loss":float(loss.detach())*TAPT_GRAD_ACCUM,
                        }
                    )

                if max_updates and update >= max_updates:
                    stop = True
                    break

        if stop:
            break

    mlm.save_pretrained(
        out_dir,
        safe_serialization=True,
    )
    TOKENIZER.save_pretrained(out_dir)

    # Explicit cleanup prevents stage-to-stage CUDA accumulation.
    opt.zero_grad(set_to_none=True)
    opt.state.clear()
    for p in mlm.parameters():
        p.grad = None

    mlm.to("cpu")
    del mlm, opt, sched, scaler, dl, ds, texts
    free_memory()

    return str(out_dir)

In [ ]:
# ============================================================
# CELL 7 — SOURCE-TRAIN-ONLY RATIONALE EMBEDDINGS
# ============================================================
def mean_pool(H, mask):
    m = mask.unsqueeze(-1).to(H.dtype)
    return (
        (H*m).sum(1)
        / m.sum(1).clamp_min(1)
    )

def build_note_embeddings(train_df, tag):
    """
    Frozen E5 embeddings for the supplied training set only.
    LODO target-domain notes are never used.
    """
    if not USE_RATIONALE_ALIGNMENT:
        return None, 0

    cache = CACHE_DIR / "rationale"
    cache.mkdir(parents=True, exist_ok=True)
    cache_file = cache / f"{tag}.npz"

    unique = train_df[
        ["id","note"]
    ].drop_duplicates("id").copy()

    unique["id"] = unique["id"].astype(str)
    unique["note"] = (
        unique["note"].fillna("").astype(str)
    )

    if cache_file.exists():
        x = np.load(
            cache_file,
            allow_pickle=True,
        )
        ids = x["ids"].astype(str).tolist()
        emb = x["emb"].astype("float32")

        assert set(ids) == set(unique.id), (
            f"Rationale cache mismatch for {tag}"
        )

        print("Using rationale cache:", cache_file)
        return {
            k:emb[i]
            for i,k in enumerate(ids)
        }, emb.shape[1]

    print(
        f"Encoding rationale notes [{tag}] | "
        f"scenarios={len(unique)}"
    )

    tok = AutoTokenizer.from_pretrained(
        RATIONALE_MODEL
    )
    enc = AutoModel.from_pretrained(
        RATIONALE_MODEL,
        add_pooling_layer=False,
        torch_dtype=(
            AMP_DTYPE
            if torch.cuda.is_available()
            else torch.float32
        ),
    ).to(DEVICE)
    enc.eval()

    notes = [
        "passage: " + x
        for x in unique.note.tolist()
    ]

    vecs = []
    bs = 4

    for i in range(0, len(notes), bs):
        b = tok(
            notes[i:i+bs],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        b = {
            k:v.to(DEVICE)
            for k,v in b.items()
        }

        with torch.no_grad(), amp_context():
            H = enc(**b).last_hidden_state

        z = F.normalize(
            mean_pool(
                H.float(),
                b["attention_mask"],
            ),
            dim=-1,
        )
        vecs.append(z.cpu().numpy())

    emb = np.concatenate(
        vecs,
        axis=0,
    ).astype("float32")

    np.savez_compressed(
        cache_file,
        ids=unique.id.values,
        emb=emb,
    )

    out = {
        k:emb[i]
        for i,k in enumerate(unique.id.tolist())
    }
    dim = int(emb.shape[1])

    enc.to("cpu")
    del enc, tok
    free_memory()

    return out, dim

In [ ]:
# ============================================================
# CELL 8 — 4-VIEW SCENARIO DATASET + AUX-SAFE COLLATE
# ============================================================
class ScenarioDataset(Dataset):
    def __init__(
        self,
        df,
        augment=False,
        seed=42,
    ):
        self.df = df.reset_index(drop=True)
        self.augment = augment
        self.seed = seed

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        rng = random.Random(
            self.seed*1000003 + i
        )

        texts = []
        for v in TEXT_VIEWS:
            t = normalize_security_text(r[v])

            if (
                self.augment
                and rng.random() < AUGMENT_PROB
            ):
                t = augment_text(
                    t,
                    v,
                    r.security_intent,
                    rng,
                )

            texts.append(t)

        return {
            "texts":texts,
            "id":str(r.id),
            "family":str(r.scenario_family_id),
            "decision":LABEL2ID[r.decision],
            "intent":INTENT2ID[r.security_intent],
            "domain":DOMAIN2ID[r.domain],
            "risk":RISK2ID[r.risk_level],
        }

def make_collate(
    training_aux=False,
    note_map=None,
):
    def collate(batch):
        texts = [
            t
            for x in batch
            for t in x["texts"]
        ]

        enc = TOKENIZER(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )

        B = len(batch)

        for k in [
            "decision",
            "intent",
            "domain",
            "risk",
        ]:
            vals = torch.tensor(
                [x[k] for x in batch],
                dtype=torch.long,
            )
            enc[k] = vals.repeat_interleave(4)

        enc["scenario"] = (
            torch.arange(
                B,
                dtype=torch.long,
            ).repeat_interleave(4)
        )

        enc["ids"] = [
            x["id"] for x in batch
        ]
        enc["families"] = [
            x["family"] for x in batch
        ]

        if (
            training_aux
            and USE_RATIONALE_ALIGNMENT
        ):
            assert note_map is not None

            missing = [
                x["id"]
                for x in batch
                if x["id"] not in note_map
            ]
            assert not missing, (
                f"Missing rationale embeddings: "
                f"{missing[:3]}"
            )

            ne = np.stack([
                note_map[x["id"]]
                for x in batch
            ])

            enc["note_emb"] = torch.tensor(
                ne,
                dtype=torch.float32,
            ).repeat_interleave(4, dim=0)

        return enc

    return collate

# Inference contract check.
_probe = ScenarioDataset(
    MAIN_VAL.head(1),
    augment=False,
)[0]

_probe_batch = make_collate(
    training_aux=False
)([_probe])

assert "note_emb" not in _probe_batch
print("Evaluation auxiliary-data isolation: PASS")

In [ ]:
# ============================================================
# CELL 9 — BANGLAGUARD XLM-R BASE ARCHITECTURE + LOSSES
# ============================================================
class BanglaGuardXLMRBase(nn.Module):
    def __init__(
        self,
        backbone,
        n_dom=3,
        n_int=7,
        n_dec=3,
        n_risk=5,
        pol_dim=64,
        rat_dim=1024,
        proj_dim=128,
        dropout=0.1,
    ):
        super().__init__()

        self.enc = AutoModel.from_pretrained(
            backbone,
            add_pooling_layer=False,
        )

        d = self.enc.config.hidden_size
        self.n_dom = n_dom

        self.attn = nn.Linear(d,1)

        self.pool = nn.Sequential(
            nn.Linear(2*d,d),
            nn.GELU(),
            nn.LayerNorm(d),
        )

        self.drop = nn.Dropout(dropout)

        self.dom_head = nn.Linear(d,n_dom)
        self.int_head = nn.Linear(d,n_int)
        self.risk_head = nn.Linear(d,n_risk-1)

        # n_dom = generic-policy index.
        self.policy = nn.Embedding(
            n_dom+1,
            pol_dim,
        )

        self.film = nn.Linear(
            pol_dim,
            2*d,
        )

        nn.init.zeros_(self.film.weight)
        nn.init.zeros_(self.film.bias)

        self.dec_head = nn.Sequential(
            nn.Linear(d+n_int,d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d,n_dec),
        )

        self.proj = nn.Sequential(
            nn.Linear(d,d),
            nn.GELU(),
            nn.Linear(d,proj_dim),
        )

        self.rat = (
            nn.Linear(d,rat_dim)
            if rat_dim > 0
            else None
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        domain_id=None,
        policy_dropout=0.0,
    ):
        H = self.enc(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state

        # AMP-safe attention pooling.
        a = self.attn(H).squeeze(-1).float()
        a = a.masked_fill(
            attention_mask == 0,
            -1e4,
        )

        Hf = H.float()

        att = (
            a.softmax(-1).unsqueeze(-1)
            * Hf
        ).sum(1)

        cls = Hf[:,0]

        h = self.drop(
            self.pool(
                torch.cat([cls,att],dim=-1)
            )
        )

        dom_logits = self.dom_head(h)
        int_logits = self.int_head(h)
        risk_logits = self.risk_head(h)

        if domain_id is not None:
            # Explicit policy index.
            # len(DOMAINS)=3 is the generic policy.
            w = F.one_hot(
                domain_id,
                self.n_dom+1,
            ).to(h.dtype)
        else:
            # Normal benchmark path:
            # policy conditioned on predicted domain.
            p_dom = dom_logits.detach().softmax(-1)
            w = torch.cat(
                [
                    p_dom,
                    p_dom.new_zeros(
                        p_dom.size(0),
                        1,
                    ),
                ],
                dim=-1,
            )

        # Train generic policy through policy dropout.
        if (
            self.training
            and policy_dropout > 0
        ):
            generic = F.one_hot(
                torch.tensor(
                    self.n_dom,
                    device=h.device,
                ),
                self.n_dom+1,
            ).to(h.dtype)

            mask = (
                torch.rand(
                    h.size(0),
                    1,
                    device=h.device,
                )
                < policy_dropout
            )

            w = torch.where(
                mask,
                generic.expand_as(w),
                w,
            )

        gamma,beta = self.film(
            w @ self.policy.weight.to(h.dtype)
        ).chunk(2, dim=-1)

        h_pol = h*(1+gamma) + beta

        dec_logits = self.dec_head(
            torch.cat(
                [
                    h_pol,
                    int_logits.softmax(-1),
                ],
                dim=-1,
            )
        )

        return {
            "h":h,
            "dec":dec_logits,
            "int":int_logits,
            "dom":dom_logits,
            "risk":risk_logits,
        }

def corn_loss(logits, y):
    losses = []
    count = 0

    for k in range(logits.size(1)):
        m = y >= k

        if m.any():
            losses.append(
                F.binary_cross_entropy_with_logits(
                    logits[m,k],
                    (y[m] > k).float(),
                    reduction="sum",
                )
            )
            count += int(m.sum())

    if not losses:
        return logits.new_tensor(0.0)

    return (
        torch.stack(losses).sum()
        / max(count,1)
    )

def corn_severity(logits):
    return (
        torch.cumprod(
            torch.sigmoid(logits),
            dim=-1,
        ).sum(-1)
        / logits.size(-1)
    )

def multiview_js(logits, n_views=4):
    ctx = (
        torch.autocast(
            "cuda",
            enabled=False,
        )
        if logits.is_cuda
        else nullcontext()
    )

    with ctx:
        x = logits.float()
        logp = (
            x.log_softmax(-1)
            .view(
                -1,
                n_views,
                x.size(-1),
            )
        )
        p = logp.exp()
        logm = (
            p.mean(
                1,
                keepdim=True,
            )
            .clamp_min(1e-8)
            .log()
        )

        return (
            p*(logp-logm)
        ).sum(-1).mean()

def supcon(
    z,
    group,
    tau=0.07,
):
    ctx = (
        torch.autocast(
            "cuda",
            enabled=False,
        )
        if z.is_cuda
        else nullcontext()
    )

    with ctx:
        zf = F.normalize(
            z.float(),
            dim=-1,
        )

        gf = group.to(zf.device)

        sim = (
            zf @ zf.T
        ) / float(tau)

        eye = torch.eye(
            len(zf),
            dtype=torch.bool,
            device=zf.device,
        )

        sim = sim.masked_fill(
            eye,
            -1e4,
        )

        pos = (
            gf[:,None] == gf[None,:]
        ) & ~eye

        denom = pos.sum(1)
        valid = denom > 0

        if not valid.any():
            return zf.new_tensor(0.0)

        logp = (
            sim
            - sim.logsumexp(
                1,
                keepdim=True,
            )
        )

        return -(
            (logp[valid]*pos[valid])
            .sum(1)
            .div(denom[valid])
            .mean()
        )

def total_loss(
    out,
    batch,
    model,
):
    ce = F.cross_entropy(
        out["dec"].float(),
        batch["decision"],
        weight=CLASS_W.to(
            out["dec"].device
        ),
        label_smoothing=LABEL_SMOOTHING,
    )

    li = F.cross_entropy(
        out["int"].float(),
        batch["intent"],
        label_smoothing=LABEL_SMOOTHING,
    )

    ld = F.cross_entropy(
        out["dom"].float(),
        batch["domain"],
    )

    lo = corn_loss(
        out["risk"].float(),
        batch["risk"],
    )

    lc = (
        multiview_js(out["dec"])
        + 0.5*multiview_js(out["int"])
    )

    ls = supcon(
        model.proj(out["h"]),
        batch["scenario"],
        SUPCON_TAU,
    )

    loss = (
        ce
        + LAMBDA_INT*li
        + LAMBDA_DOM*ld
        + LAMBDA_ORD*lo
        + LAMBDA_CONS*lc
        + LAMBDA_SCL*ls
    )

    if USE_RATIONALE_ALIGNMENT:
        rat = F.cosine_similarity(
            model.rat(out["h"]).float(),
            batch["note_emb"],
            dim=-1,
        )

        loss = (
            loss
            + LAMBDA_RAT*(1-rat).mean()
        )

    return loss

print("BanglaGuard-Base architecture ready.")

In [ ]:
# ============================================================
# CELL 10 — AMP REGRESSION TEST
# ============================================================
_z = torch.randn(
    16,
    128,
    device=DEVICE,
    dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
)

_g = (
    torch.arange(
        4,
        device=DEVICE,
    ).repeat_interleave(4)
)

_amp_loss = supcon(
    _z,
    _g,
    0.07,
)

assert torch.isfinite(_amp_loss)

print(
    "AMP SupCon regression test: PASS | loss =",
    float(_amp_loss)
)

del _z, _g, _amp_loss
free_memory()

In [ ]:
# ============================================================
# CELL 11 — METRICS + DOMAIN-WISE ANALYSIS
# ============================================================
def ece_score(
    y,
    probs,
    n_bins=15,
):
    y = np.asarray(y)
    probs = np.asarray(probs)

    pred = probs.argmax(1)
    conf = probs.max(1)
    cor = (pred == y).astype(float)

    out = 0.0
    edges = np.linspace(
        0,
        1,
        n_bins+1,
    )

    for lo,hi in zip(
        edges[:-1],
        edges[1:],
    ):
        m = (
            (conf > lo)
            & (conf <= hi)
        )

        if m.any():
            out += (
                m.mean()
                * abs(
                    cor[m].mean()
                    - conf[m].mean()
                )
            )

    return float(out)

def reliability_table(
    y,
    probs,
    n_bins=15,
):
    y = np.asarray(y)
    probs = np.asarray(probs)

    pred = probs.argmax(1)
    conf = probs.max(1)
    cor = (pred == y).astype(float)

    rows = []
    edges = np.linspace(
        0,
        1,
        n_bins+1,
    )

    for b,(lo,hi) in enumerate(
        zip(
            edges[:-1],
            edges[1:],
        )
    ):
        m = (
            (conf > lo)
            & (conf <= hi)
        )

        rows.append({
            "bin":b,
            "lo":lo,
            "hi":hi,
            "n":int(m.sum()),
            "mean_confidence":(
                float(conf[m].mean())
                if m.any()
                else np.nan
            ),
            "accuracy":(
                float(cor[m].mean())
                if m.any()
                else np.nan
            ),
        })

    return pd.DataFrame(rows)

def multiclass_brier(y, probs):
    oh = np.eye(3)[
        np.asarray(
            y,
            dtype=int,
        )
    ]

    return float(
        np.mean(
            np.sum(
                (probs-oh)**2,
                axis=1,
            )
        )
    )

def metric_dict(
    y,
    probs,
    hard_negative=None,
):
    y = np.asarray(y,dtype=int)
    probs = np.asarray(probs)
    p = probs.argmax(1)

    out = {
        "accuracy":float(
            accuracy_score(y,p)
        ),
        "balanced_accuracy":float(
            balanced_accuracy_score(y,p)
        ),
        "macro_f1":float(
            f1_score(
                y,
                p,
                average="macro",
                labels=[0,1,2],
                zero_division=0,
            )
        ),
        "weighted_f1":float(
            f1_score(
                y,
                p,
                average="weighted",
                labels=[0,1,2],
                zero_division=0,
            )
        ),
        "mcc":float(
            matthews_corrcoef(y,p)
        ),
        "ece":ece_score(
            y,
            probs,
        ),
        "brier":multiclass_brier(
            y,
            probs,
        ),
    }

    pr,rc,f1,sup = (
        precision_recall_fscore_support(
            y,
            p,
            labels=[0,1,2],
            zero_division=0,
        )
    )

    for i,lab in enumerate(LABELS):
        out[f"precision_{lab}"] = float(pr[i])
        out[f"recall_{lab}"] = float(rc[i])
        out[f"f1_{lab}"] = float(f1[i])
        out[f"support_{lab}"] = int(sup[i])

    yb = (
        y != LABEL2ID["ALLOW"]
    ).astype(int)

    pb = (
        p != LABEL2ID["ALLOW"]
    ).astype(int)

    p_threat = (
        probs[:,LABEL2ID["REVIEW"]]
        + probs[:,LABEL2ID["BLOCK"]]
    )

    out["binary_macro_f1"] = float(
        f1_score(
            yb,
            pb,
            average="macro",
            zero_division=0,
        )
    )

    out["threat_recall"] = float(
        (
            (pb==1)
            & (yb==1)
        ).sum()
        / max(
            (yb==1).sum(),
            1,
        )
    )

    out["threat_fnr"] = (
        1.0
        - out["threat_recall"]
    )

    try:
        out["threat_auroc"] = float(
            roc_auc_score(
                yb,
                p_threat,
            )
        )

        out["threat_pr_auc"] = float(
            average_precision_score(
                yb,
                p_threat,
            )
        )

    except Exception:
        out["threat_auroc"] = np.nan
        out["threat_pr_auc"] = np.nan

    if hard_negative is not None:
        h = np.asarray(
            hard_negative,
            dtype=bool,
        )

        out["hard_negative_over_refusal"] = (
            float(
                (
                    p[h]
                    != LABEL2ID["ALLOW"]
                ).mean()
            )
            if h.any()
            else np.nan
        )

    return out

def summarize_predictions(df):
    rows = []

    def add(
        slice_type,
        slice_value,
        g,
    ):
        m = metric_dict(
            g.y_true.values,
            g[
                [
                    f"p_{x}"
                    for x in LABELS
                ]
            ].values,
            g.hard_negative.values,
        )

        m.update({
            "slice_type":slice_type,
            "slice_value":str(slice_value),
            "n":len(g),
        })

        rows.append(m)

    add(
        "overall",
        "all",
        df,
    )

    for col in [
        "text_view",
        "domain",
        "decision_true",
        "security_intent",
    ]:
        for v,g in df.groupby(col):
            add(
                col,
                v,
                g,
            )

    matched = df[
        df.in_decision_matched_test.astype(bool)
    ]

    if len(matched):
        add(
            "decision_matched",
            "true",
            matched,
        )

    return pd.DataFrame(rows)

def intent_metrics(df):
    if "intent_pred_id" not in df:
        return pd.DataFrame()

    y = df.intent_true_id.values.astype(int)
    p = df.intent_pred_id.values.astype(int)

    rows = [{
        "slice_type":"overall",
        "slice_value":"all",
        "n":len(df),
        "intent_macro_f1":float(
            f1_score(
                y,
                p,
                average="macro",
                labels=list(range(7)),
                zero_division=0,
            )
        ),
        "intent_accuracy":float(
            accuracy_score(y,p)
        ),
    }]

    pr,rc,f1,sup = (
        precision_recall_fscore_support(
            y,
            p,
            labels=list(range(7)),
            zero_division=0,
        )
    )

    for i,name in enumerate(INTENTS):
        rows.append({
            "slice_type":"intent_class",
            "slice_value":name,
            "n":int(sup[i]),
            "precision":float(pr[i]),
            "recall":float(rc[i]),
            "f1":float(f1[i]),
        })

    return pd.DataFrame(rows)

def family_bootstrap(
    df,
    n_boot=N_BOOT,
    seed=42,
):
    fams = np.asarray(
        df.scenario_family_id.unique(),
        dtype=object,
    )

    fam_to_idx = {
        f:np.where(
            df.scenario_family_id.values == f
        )[0]
        for f in fams
    }

    y = df.y_true.values
    probs = df[
        [
            f"p_{x}"
            for x in LABELS
        ]
    ].values
    hn = df.hard_negative.values

    rng = np.random.default_rng(seed)
    vals = []

    for _ in range(n_boot):
        sampled = rng.choice(
            fams,
            size=len(fams),
            replace=True,
        )

        idx = np.concatenate([
            fam_to_idx[f]
            for f in sampled
        ])

        vals.append(
            metric_dict(
                y[idx],
                probs[idx],
                hn[idx],
            )["macro_f1"]
        )

    return [
        float(x)
        for x in np.percentile(
            vals,
            [2.5,50,97.5],
        )
    ]

def domain_metrics(df):
    rows = []

    for domain,g in df.groupby("domain"):
        m = metric_dict(
            g.y_true.values,
            g[
                [
                    f"p_{x}"
                    for x in LABELS
                ]
            ].values,
            g.hard_negative.values,
        )

        m.update({
            "domain":domain,
            "n_prompts":len(g),
            "n_scenarios":g.id.nunique(),
        })

        rows.append(m)

    return pd.DataFrame(rows)

def domain_view_metrics(df):
    rows = []

    for (domain,view),g in df.groupby(
        ["domain","text_view"]
    ):
        m = metric_dict(
            g.y_true.values,
            g[
                [
                    f"p_{x}"
                    for x in LABELS
                ]
            ].values,
            g.hard_negative.values,
        )

        rows.append({
            "domain":domain,
            "text_view":view,
            "n":len(g),
            "macro_f1":m["macro_f1"],
            "f1_ALLOW":m["f1_ALLOW"],
            "f1_REVIEW":m["f1_REVIEW"],
            "f1_BLOCK":m["f1_BLOCK"],
            "threat_recall":m["threat_recall"],
            "hard_negative_over_refusal":m.get(
                "hard_negative_over_refusal",
                np.nan,
            ),
        })

    return pd.DataFrame(rows)

print("Metric and domain-analysis functions ready.")

In [ ]:
# ============================================================
# CELL 12 — PREDICTION + CALIBRATION + CONFORMAL
# ============================================================
@torch.no_grad()
def predict(
    model,
    df,
    batch_size=EVAL_SCENARIO_BATCH,
    domain_id_override=None,
):
    model.eval()

    ds = ScenarioDataset(
        df,
        augment=False,
    )

    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=make_collate(
            training_aux=False
        ),
    )

    recs = []
    offset = 0

    for batch in dl:
        ids = batch.pop("ids")
        fams = batch.pop("families")

        batch.pop("scenario",None)
        batch.pop("note_emb",None)
        batch.pop("decision",None)
        batch.pop("intent",None)
        batch.pop("domain",None)
        batch.pop("risk",None)

        x = {
            k:batch[k].to(DEVICE)
            for k in [
                "input_ids",
                "attention_mask",
            ]
        }

        did = None

        if domain_id_override is not None:
            did = torch.full(
                (
                    x["input_ids"].shape[0],
                ),
                int(domain_id_override),
                dtype=torch.long,
                device=DEVICE,
            )

        with amp_context():
            out = model(
                input_ids=x["input_ids"],
                attention_mask=x["attention_mask"],
                domain_id=did,
                policy_dropout=0.0,
            )

        probs = (
            torch.softmax(
                out["dec"].float(),
                dim=-1,
            )
            .cpu()
            .numpy()
        )

        ip = (
            torch.softmax(
                out["int"].float(),
                dim=-1,
            )
            .cpu()
            .numpy()
        )

        dp = (
            torch.softmax(
                out["dom"].float(),
                dim=-1,
            )
            .cpu()
            .numpy()
        )

        sev = (
            corn_severity(
                out["risk"].float()
            )
            .cpu()
            .numpy()
        )

        B = len(ids)

        for bi in range(B):
            src = df.iloc[offset+bi]

            for vi,v in enumerate(TEXT_VIEWS):
                j = bi*4 + vi
                pred = int(
                    probs[j].argmax()
                )

                recs.append({
                    "id":ids[bi],
                    "scenario_family_id":fams[bi],
                    "domain":src.domain,
                    "security_intent":src.security_intent,
                    "decision_true":src.decision,
                    "risk_level":src.risk_level,
                    "hard_negative_type":src.hard_negative_type,
                    "hard_negative":(
                        src.hard_negative_type
                        == "hard_negative"
                    ),
                    "in_decision_matched_test":bool(
                        src.in_decision_matched_test
                    ),
                    "text_view":v,
                    "y_true":LABEL2ID[src.decision],
                    "p_ALLOW":float(probs[j,0]),
                    "p_REVIEW":float(probs[j,1]),
                    "p_BLOCK":float(probs[j,2]),
                    "p_threat":float(
                        probs[j,1]
                        + probs[j,2]
                    ),
                    "pred_id":pred,
                    "pred_decision":ID2LABEL[pred],
                    "intent_true_id":INTENT2ID[
                        src.security_intent
                    ],
                    "intent_pred_id":int(
                        ip[j].argmax()
                    ),
                    "domain_pred_id":int(
                        dp[j].argmax()
                    ),
                    "severity":float(sev[j]),
                })

        offset += B

    assert len(recs) == len(df)*4
    return pd.DataFrame(recs)

def family_val_halves(df):
    strata = (
        df.domain.astype(str)
        + "|"
        + df.decision.astype(str)
    )

    sg = StratifiedGroupKFold(
        n_splits=2,
        shuffle=True,
        random_state=2026,
    )

    a,b = next(
        sg.split(
            df,
            y=strata,
            groups=df.scenario_family_id,
        )
    )

    A = df.iloc[a].reset_index(drop=True)
    B = df.iloc[b].reset_index(drop=True)

    assert set(
        A.scenario_family_id
    ).isdisjoint(
        set(B.scenario_family_id)
    )

    return A,B

def fit_temperature(pred_df):
    p = pred_df[
        [
            f"p_{x}"
            for x in LABELS
        ]
    ].values.clip(1e-8,1)

    logits = torch.tensor(
        np.log(p),
        dtype=torch.float32,
    )

    y = torch.tensor(
        pred_df.y_true.values,
        dtype=torch.long,
    )

    log_t = torch.zeros(
        1,
        requires_grad=True,
    )

    opt = torch.optim.LBFGS(
        [log_t],
        lr=0.05,
        max_iter=150,
    )

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(
            logits/log_t.exp(),
            y,
        )
        loss.backward()
        return loss

    opt.step(closure)

    return float(
        log_t.exp().detach()
    )

def apply_temperature(
    pred_df,
    T,
):
    out = pred_df.copy()

    p = out[
        [
            f"p_{x}"
            for x in LABELS
        ]
    ].values.clip(1e-8,1)

    z = np.log(p) / max(T,1e-6)
    z -= z.max(1,keepdims=True)

    q = np.exp(z)
    q /= q.sum(1,keepdims=True)

    for i,l in enumerate(LABELS):
        out[f"p_{l}"] = q[:,i]

    out["p_threat"] = (
        out.p_REVIEW
        + out.p_BLOCK
    )

    out["pred_id"] = q.argmax(1)
    out["pred_decision"] = [
        ID2LABEL[int(i)]
        for i in out.pred_id
    ]

    return out

def fit_mondrian_conformal(
    pred_df,
    alpha=0.05,
):
    probs = pred_df[
        [
            f"p_{x}"
            for x in LABELS
        ]
    ].values

    y = pred_df.y_true.values.astype(int)

    q = np.ones(3)

    for c in range(3):
        s = np.sort(
            1.0
            - probs[y==c,c]
        )

        if len(s)==0:
            continue

        k = (
            int(
                np.ceil(
                    (len(s)+1)
                    * (1-alpha)
                )
            )
            - 1
        )

        k = min(
            max(k,0),
            len(s)-1,
        )

        q[c] = s[k]

    return q

def conformal_decisions(
    probs,
    q,
):
    out = []
    ALLOW,REVIEW,BLOCK = 0,1,2

    for p in probs:
        in_set = [
            1-p[c] <= q[c]
            for c in range(3)
        ]

        a = int(np.argmax(p))

        if in_set[BLOCK]:
            out.append(
                BLOCK
                if a == BLOCK
                else REVIEW
            )
        else:
            out.append(
                REVIEW
                if a == BLOCK
                else a
            )

    return np.asarray(out)

In [ ]:
# ============================================================
# CELL 13 — PARAMETERIZED MEMORY-SAFE TRAINING
# ============================================================
def build_model(
    backbone,
    rat_dim,
):
    seed_all(FIXED_HEAD_INIT_SEED)

    model = BanglaGuardXLMRBase(
        backbone,
        rat_dim=(
            rat_dim
            if USE_RATIONALE_ALIGNMENT
            else 0
        ),
        dropout=DROPOUT,
    )

    if hasattr(
        model.enc,
        "gradient_checkpointing_enable",
    ):
        model.enc.gradient_checkpointing_enable()

    return model.to(DEVICE)

def optimizer_groups(model):
    n_layers = int(
        getattr(
            model.enc.config,
            "num_hidden_layers",
            12,
        )
    )

    groups = {}

    for name,p in model.named_parameters():
        if not p.requires_grad:
            continue

        wd = (
            0.0
            if (
                "bias" in name
                or "LayerNorm" in name
                or "layer_norm" in name.lower()
            )
            else WEIGHT_DECAY
        )

        if not name.startswith("enc."):
            lr = LR_HEADS
        else:
            mm = re.search(
                r"encoder\.layer\.(\d+)",
                name,
            )

            if mm:
                layer = int(mm.group(1))
                lr = (
                    LR_ENCODER
                    * (
                        LAYERWISE_LR_DECAY
                        ** (
                            n_layers
                            - 1
                            - layer
                        )
                    )
                )

            elif "embeddings" in name:
                lr = (
                    LR_ENCODER
                    * (
                        LAYERWISE_LR_DECAY
                        ** n_layers
                    )
                )

            else:
                lr = LR_ENCODER

        key = (
            round(lr,12),
            wd,
        )

        groups.setdefault(
            key,
            [],
        ).append(p)

    return [
        {
            "params":ps,
            "lr":lr,
            "weight_decay":wd,
        }
        for (lr,wd),ps
        in groups.items()
    ]

def train_seed(
    train_df,
    val_df,
    active_backbone,
    note_map,
    rat_dim,
    seed,
    tag,
):
    free_memory()

    model = build_model(
        active_backbone,
        rat_dim,
    )

    seed_all(seed)

    ds = ScenarioDataset(
        train_df,
        augment=True,
        seed=seed,
    )

    gen = torch.Generator()
    gen.manual_seed(seed)

    dl = DataLoader(
        ds,
        batch_size=TRAIN_SCENARIO_BATCH,
        shuffle=True,
        collate_fn=make_collate(
            training_aux=True,
            note_map=note_map,
        ),
        generator=gen,
    )

    # foreach=False avoids multi-tensor Adam peak-memory spikes.
    opt = torch.optim.AdamW(
        optimizer_groups(model),
        foreach=False,
    )

    epochs = (
        1
        if SMOKE_RUN
        else EPOCHS
    )

    steps_per_epoch = max(
        1,
        math.ceil(
            len(dl)/GRAD_ACCUM
        ),
    )

    total_steps = (
        SMOKE_MAX_STEPS
        if SMOKE_RUN
        else steps_per_epoch*epochs
    )

    sched = get_linear_schedule_with_warmup(
        opt,
        int(
            WARMUP_RATIO
            * total_steps
        ),
        total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_SCALER
    )

    best = -1.0
    best_state = None
    bad = 0
    step = 0
    history = []
    t0 = time.time()

    for ep in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)
        losses = []

        for bi,batch in enumerate(dl):
            batch.pop("ids")
            batch.pop("families")

            batch = {
                k:(
                    v.to(DEVICE)
                    if torch.is_tensor(v)
                    else v
                )
                for k,v in batch.items()
            }

            with amp_context():
                out = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    policy_dropout=POLICY_DROPOUT,
                )

                loss = total_loss(
                    out,
                    batch,
                    model,
                )

                loss = (
                    loss
                    / GRAD_ACCUM
                )

            scaler.scale(loss).backward()

            losses.append(
                float(loss.detach())
                * GRAD_ACCUM
            )

            # Backward has consumed the graph.
            del out, loss

            if (
                (bi+1)%GRAD_ACCUM == 0
                or (bi+1) == len(dl)
            ):
                scaler.unscale_(opt)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM,
                )

                scaler.step(opt)
                scaler.update()

                sched.step()
                opt.zero_grad(set_to_none=True)

                step += 1

                if (
                    SMOKE_RUN
                    and step >= SMOKE_MAX_STEPS
                ):
                    break

        val_pred = predict(
            model,
            val_df,
        )

        vf1 = f1_score(
            val_pred.y_true,
            val_pred.pred_id,
            average="macro",
            labels=[0,1,2],
            zero_division=0,
        )

        row = {
            "tag":tag,
            "seed":seed,
            "epoch":ep+1,
            "step":step,
            "train_loss":float(
                np.mean(losses)
            ),
            "val_macro_f1":float(vf1),
            "elapsed_min":float(
                (time.time()-t0)/60
            ),
        }

        history.append(row)
        print(row)

        if vf1 > best + EARLY_STOP_MIN_DELTA:
            best = float(vf1)
            bad = 0

            best_state = {
                k:v.detach().cpu().clone()
                for k,v
                in model.state_dict().items()
            }

        else:
            bad += 1

        del val_pred

        if SMOKE_RUN:
            break

        if bad >= PATIENCE:
            print(
                f"Early stop [{tag}] seed={seed}"
            )
            break

    assert best_state is not None
    model.load_state_dict(
        best_state,
        strict=True,
    )

    # Free Adam moments before evaluation / next run.
    opt.zero_grad(set_to_none=True)
    opt.state.clear()
    del opt, sched, scaler, dl, ds
    free_memory()

    return (
        model,
        pd.DataFrame(history),
        best,
        best_state,
    )

print("Training utilities ready.")

In [ ]:
# ============================================================
# CELL 14 — EFFICIENCY + UNSEEN ROBUSTNESS
# ============================================================
@torch.no_grad()
def measure_efficiency(
    model,
    text,
    n_rep=50,
):
    model.eval()

    e = TOKENIZER(
        [
            normalize_security_text(text)
        ],
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )

    e = {
        k:v.to(DEVICE)
        for k,v in e.items()
    }

    for _ in range(10):
        with amp_context():
            model(
                input_ids=e["input_ids"],
                attention_mask=e["attention_mask"],
            )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []

    for _ in range(n_rep):
        t0 = time.perf_counter()

        with amp_context():
            model(
                input_ids=e["input_ids"],
                attention_mask=e["attention_mask"],
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        times.append(
            (
                time.perf_counter()
                - t0
            )*1000
        )

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    return {
        "params":int(total),
        "checkpoint_estimate_fp16_mb":float(
            total*2/1024**2
        ),
        "latency_p50_ms":float(
            np.percentile(times,50)
        ),
        "latency_p95_ms":float(
            np.percentile(times,95)
        ),
        "peak_gpu_mem_gb":(
            float(
                torch.cuda.max_memory_allocated()
                / 1024**3
            )
            if torch.cuda.is_available()
            else 0.0
        ),
    }

EMOJIS = [
    "🔒","⚠️","🧾","📌","🔐"
]

def unseen_stress_transform(
    text,
    seed,
):
    rng = random.Random(seed)
    t = str(text)

    mode = rng.choice([
        "emoji",
        "case_flip",
        "leet",
        "xml_wrap",
        "newline",
    ])

    if mode == "emoji":
        toks = t.split()
        pos = (
            rng.randrange(len(toks)+1)
            if toks
            else 0
        )
        toks.insert(
            pos,
            rng.choice(EMOJIS),
        )
        return " ".join(toks)

    if mode == "case_flip":
        return "".join(
            (
                c.upper()
                if (
                    c.isascii()
                    and c.isalpha()
                    and rng.random() < 0.5
                )
                else c
            )
            for c in t
        )

    if mode == "leet":
        trans = str.maketrans({
            "a":"4",
            "e":"3",
            "i":"1",
            "o":"0",
            "s":"5",
        })

        toks = t.split()
        cand = [
            i
            for i,x in enumerate(toks)
            if x.isascii()
            and x.isalpha()
        ]

        if cand:
            j = rng.choice(cand)
            toks[j] = (
                toks[j]
                .lower()
                .translate(trans)
            )

        return " ".join(toks)

    if mode == "xml_wrap":
        return "<content>" + t + "</content>"

    if mode == "newline":
        toks = t.split()

        if len(toks) > 2:
            j = rng.randrange(
                1,
                len(toks),
            )

            return (
                " ".join(toks[:j])
                + "\n"
                + " ".join(toks[j:])
            )

        return t + "\n"

    return t

def build_unseen_stress_df(
    df,
    seed=2026,
):
    out = df.copy()

    for ri in range(len(out)):
        for vi,v in enumerate(TEXT_VIEWS):
            out.at[
                out.index[ri],
                v,
            ] = unseen_stress_transform(
                out.iloc[ri][v],
                seed + ri*17 + vi,
            )

    return out

In [ ]:
# ============================================================
# CELL 15 — STAGE A: FULL MAIN BANGLAGUARD-BASE EXPERIMENT
# ============================================================
MAIN_DOMAIN_FINAL = pd.DataFrame()
MAIN_DOMAIN_VIEW_FINAL = pd.DataFrame()
MAIN_REPORT = {}

if RUN_MAIN_EXPERIMENT:
    print("\n" + "="*78)
    print("STAGE A — FULL MAIN BENCHMARK")
    print("="*78)

    main_seeds = (
        [42]
        if SMOKE_RUN
        else MAIN_SEEDS
    )

    main_backbone = prepare_tapt_backbone(
        MAIN_TRAIN,
        "main_all_domains",
    )

    main_note_map, main_rat_dim = (
        build_note_embeddings(
            MAIN_TRAIN,
            "main_all_domains",
        )
    )

    state_dir = (
        MAIN_DIR
        / "_seed_states"
    )
    state_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    state_paths = []
    seed_rows = []

    for seed in main_seeds:
        print(
            "\n================ MAIN SEED",
            seed,
            "================"
        )

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        model,hist,vf1,state = train_seed(
            MAIN_TRAIN,
            MAIN_VAL,
            main_backbone,
            main_note_map,
            main_rat_dim,
            seed,
            "main",
        )

        hist.to_csv(
            MAIN_DIR/f"history_seed{seed}.csv",
            index=False,
        )

        state_path = (
            state_dir
            / f"state_seed{seed}.pt"
        )
        torch.save(
            state,
            state_path,
        )
        state_paths.append(state_path)
        del state
        gc.collect()

        raw = predict(
            model,
            MAIN_TEST,
        )

        raw.to_csv(
            MAIN_DIR/f"predictions_raw_seed{seed}.csv",
            index=False,
        )

        raw_metrics = summarize_predictions(raw)
        raw_metrics["seed"] = seed

        raw_metrics.to_csv(
            MAIN_DIR/f"metrics_raw_seed{seed}.csv",
            index=False,
        )

        ov = raw_metrics[
            raw_metrics.slice_type == "overall"
        ].iloc[0]

        seed_rows.append({
            "seed":seed,
            "val_macro_f1":vf1,
            "raw_test_macro_f1":float(
                ov.macro_f1
            ),
            "raw_review_f1":float(
                ov.f1_REVIEW
            ),
            "raw_over_refusal":float(
                ov.hard_negative_over_refusal
            ),
            "peak_gpu_mem_gb":(
                float(
                    torch.cuda.max_memory_allocated()
                    / 1024**3
                )
                if torch.cuda.is_available()
                else 0.0
            ),
        })

        del model, raw, raw_metrics, hist
        free_memory()

    main_seed_summary = pd.DataFrame(
        seed_rows
    )

    main_seed_summary.to_csv(
        MAIN_DIR/"seed_summary.csv",
        index=False,
    )

    display(main_seed_summary)

    # --------------------------------------------------------
    # Final main model: disk-backed weight soup.
    # --------------------------------------------------------
    if (
        USE_WEIGHT_SOUP
        and len(state_paths) > 1
    ):
        print(
            "Building main weight soup from",
            len(state_paths),
            "states"
        )

        soup = torch.load(
            state_paths[0],
            map_location="cpu",
        )

        for k,v in list(soup.items()):
            if torch.is_floating_point(v):
                soup[k] = v.float()

        for path in state_paths[1:]:
            st = torch.load(
                path,
                map_location="cpu",
            )

            for k in soup:
                if torch.is_floating_point(
                    soup[k]
                ):
                    soup[k].add_(
                        st[k].float()
                    )

            del st
            gc.collect()

        for k in soup:
            if torch.is_floating_point(
                soup[k]
            ):
                soup[k].div_(
                    len(state_paths)
                )

        free_memory()

        final_model = build_model(
            main_backbone,
            main_rat_dim,
        )

        final_model.load_state_dict(
            soup,
            strict=True,
        )

        del soup
        final_tag = "weight_soup"

    else:
        best_i = int(
            np.argmax(
                main_seed_summary
                .val_macro_f1
                .values
            )
        )

        chosen = torch.load(
            state_paths[best_i],
            map_location="cpu",
        )

        free_memory()

        final_model = build_model(
            main_backbone,
            main_rat_dim,
        )

        final_model.load_state_dict(
            chosen,
            strict=True,
        )

        del chosen
        final_tag = (
            f"seed_{main_seeds[best_i]}"
        )

    # Remove temporary seed-state files once final model exists.
    shutil.rmtree(
        state_dir,
        ignore_errors=True,
    )
    gc.collect()

    # --------------------------------------------------------
    # Calibration.
    # --------------------------------------------------------
    valA,valB = family_val_halves(
        MAIN_VAL
    )

    predA = predict(
        final_model,
        valA,
    )

    predB = predict(
        final_model,
        valB,
    )

    T_MAIN = fit_temperature(predA)

    predB_cal = apply_temperature(
        predB,
        T_MAIN,
    )

    main_test_pred = apply_temperature(
        predict(
            final_model,
            MAIN_TEST,
        ),
        T_MAIN,
    )

    main_test_metrics = summarize_predictions(
        main_test_pred
    )

    main_test_intent = intent_metrics(
        main_test_pred
    )

    main_ci = family_bootstrap(
        main_test_pred,
        N_BOOT,
        42,
    )

    main_test_pred.to_csv(
        MAIN_DIR/"predictions_final.csv",
        index=False,
    )

    main_test_metrics.to_csv(
        MAIN_DIR/"metrics_final_all_slices.csv",
        index=False,
    )

    main_test_intent.to_csv(
        MAIN_DIR/"intent_metrics_final.csv",
        index=False,
    )

    reliability_table(
        main_test_pred.y_true.values,
        main_test_pred[
            [
                f"p_{x}"
                for x in LABELS
            ]
        ].values,
        n_bins=15,
    ).to_csv(
        MAIN_DIR/"reliability_15bins.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Explicit domain-wise analysis.
    # --------------------------------------------------------
    MAIN_DOMAIN_FINAL = domain_metrics(
        main_test_pred
    )

    MAIN_DOMAIN_FINAL.to_csv(
        MAIN_DIR/"domain_metrics_final.csv",
        index=False,
    )

    MAIN_DOMAIN_VIEW_FINAL = (
        domain_view_metrics(
            main_test_pred
        )
    )

    MAIN_DOMAIN_VIEW_FINAL.to_csv(
        MAIN_DIR/"domain_by_view_metrics_final.csv",
        index=False,
    )

    print("\nMAIN DOMAIN-WISE RESULTS")
    display(
        MAIN_DOMAIN_FINAL[
            [
                "domain",
                "n_scenarios",
                "macro_f1",
                "f1_ALLOW",
                "f1_REVIEW",
                "f1_BLOCK",
                "threat_recall",
                "hard_negative_over_refusal",
                "ece",
            ]
        ]
    )

    # --------------------------------------------------------
    # Decision-matched diagnostic.
    # --------------------------------------------------------
    main_matched_macro = np.nan

    if len(MAIN_MATCHED):
        mp = apply_temperature(
            predict(
                final_model,
                MAIN_MATCHED,
            ),
            T_MAIN,
        )

        mm = summarize_predictions(mp)

        mp.to_csv(
            MAIN_DIR/"predictions_decision_matched.csv",
            index=False,
        )

        mm.to_csv(
            MAIN_DIR/"metrics_decision_matched.csv",
            index=False,
        )

        main_matched_macro = float(
            mm[
                mm.slice_type == "overall"
            ].iloc[0].macro_f1
        )

    # --------------------------------------------------------
    # Conformal alpha sweep.
    # --------------------------------------------------------
    conformal_rows = []

    if USE_CONFORMAL:
        probs = main_test_pred[
            [
                f"p_{x}"
                for x in LABELS
            ]
        ].values

        y = main_test_pred.y_true.values.astype(int)
        hard = main_test_pred.hard_negative.values.astype(bool)
        true_block = (
            y == LABEL2ID["BLOCK"]
        )

        for alpha in [
            0.01,
            0.02,
            0.05,
            0.10,
        ]:
            q = fit_mondrian_conformal(
                predB_cal,
                alpha,
            )

            cp = conformal_decisions(
                probs,
                q,
            )

            conformal_rows.append({
                "alpha":alpha,
                "q_ALLOW":float(q[0]),
                "q_REVIEW":float(q[1]),
                "q_BLOCK":float(q[2]),
                "block_to_allow_rate":(
                    float(
                        (
                            cp[true_block]
                            == LABEL2ID["ALLOW"]
                        ).mean()
                    )
                    if true_block.any()
                    else np.nan
                ),
                "review_rate":float(
                    (
                        cp
                        == LABEL2ID["REVIEW"]
                    ).mean()
                ),
                "hard_negative_over_refusal":(
                    float(
                        (
                            cp[hard]
                            != LABEL2ID["ALLOW"]
                        ).mean()
                    )
                    if hard.any()
                    else np.nan
                ),
            })

    pd.DataFrame(
        conformal_rows
    ).to_csv(
        MAIN_DIR/"conformal_alpha_sweep.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Robustness.
    # --------------------------------------------------------
    robust_macro = np.nan

    if RUN_ROBUSTNESS_STRESS:
        stress_df = build_unseen_stress_df(
            MAIN_TEST,
            2026,
        )

        stress_pred = apply_temperature(
            predict(
                final_model,
                stress_df,
            ),
            T_MAIN,
        )

        stress_metrics = summarize_predictions(
            stress_pred
        )

        stress_pred.to_csv(
            MAIN_DIR/"predictions_unseen_stress.csv",
            index=False,
        )

        stress_metrics.to_csv(
            MAIN_DIR/"metrics_unseen_stress.csv",
            index=False,
        )

        robust_macro = float(
            stress_metrics[
                stress_metrics.slice_type
                == "overall"
            ].iloc[0].macro_f1
        )

    # --------------------------------------------------------
    # Efficiency.
    # --------------------------------------------------------
    efficiency = measure_efficiency(
        final_model,
        MAIN_TEST.iloc[0]["bangla"],
        n_rep=(
            10
            if SMOKE_RUN
            else 50
        ),
    )

    pd.DataFrame([
        efficiency
    ]).to_csv(
        MAIN_DIR/"efficiency.csv",
        index=False,
    )

    # Optional deployment checkpoint.
    if SAVE_FINAL_MODEL:
        final_dir = (
            MAIN_DIR
            / "final_model"
        )

        final_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        final_model.enc.save_pretrained(
            final_dir/"encoder",
            safe_serialization=True,
        )

        TOKENIZER.save_pretrained(
            final_dir/"tokenizer"
        )

        torch.save(
            {
                "attn":final_model.attn.state_dict(),
                "pool":final_model.pool.state_dict(),
                "dom_head":final_model.dom_head.state_dict(),
                "int_head":final_model.int_head.state_dict(),
                "risk_head":final_model.risk_head.state_dict(),
                "policy":final_model.policy.state_dict(),
                "film":final_model.film.state_dict(),
                "dec_head":final_model.dec_head.state_dict(),
                "proj":final_model.proj.state_dict(),
                "rat":(
                    final_model.rat.state_dict()
                    if final_model.rat is not None
                    else None
                ),
                "temperature":T_MAIN,
            },
            final_dir/"banglaguard_base_heads.pt",
        )

    main_overall = main_test_metrics[
        main_test_metrics.slice_type
        == "overall"
    ].iloc[0]

    MAIN_REPORT = {
        "architecture":"BanglaGuard-Base / XLM-R Base",
        "stage":"main benchmark",
        "seeds":main_seeds,
        "final_model":final_tag,
        "tapt_tlm":USE_TAPT_TLM,
        "rationale_alignment_train_only":USE_RATIONALE_ALIGNMENT,
        "kd":False,
        "overall_macro_f1":float(
            main_overall.macro_f1
        ),
        "review_f1":float(
            main_overall.f1_REVIEW
        ),
        "decision_matched_macro_f1":main_matched_macro,
        "unseen_stress_macro_f1":robust_macro,
        "binary_macro_f1":float(
            main_overall.binary_macro_f1
        ),
        "threat_recall":float(
            main_overall.threat_recall
        ),
        "threat_auroc":float(
            main_overall.threat_auroc
        ),
        "threat_pr_auc":float(
            main_overall.threat_pr_auc
        ),
        "hard_negative_over_refusal":float(
            main_overall.hard_negative_over_refusal
        ),
        "ece":float(
            main_overall.ece
        ),
        "temperature":float(T_MAIN),
        "bootstrap_macro_f1_95ci":main_ci,
        "efficiency":efficiency,
    }

    with open(
        MAIN_DIR/"run_report.json",
        "w",
    ) as f:
        json.dump(
            MAIN_REPORT,
            f,
            indent=2,
        )

    print("\nMAIN REPORT")
    print(
        json.dumps(
            MAIN_REPORT,
            indent=2,
        )
    )

    # Final model is not required by LODO.
    del final_model
    del main_note_map
    free_memory()

else:
    print("Main benchmark disabled.")

In [ ]:
# ============================================================
# CELL 16 — STRICT LODO SPLIT BUILDER
# ============================================================
def build_lodo_split(target_domain):
    source_domains = [
        d
        for d in DOMAINS
        if d != target_domain
    ]

    train = MASTER_FULL[
        (MASTER_FULL.split == "train")
        & (
            MASTER_FULL.domain
            .isin(source_domains)
        )
    ].copy()

    val = MASTER_FULL[
        (MASTER_FULL.split == "val")
        & (
            MASTER_FULL.domain
            .isin(source_domains)
        )
    ].copy()

    test = MASTER_FULL[
        (MASTER_FULL.split == "heldout_test")
        & (
            MASTER_FULL.domain
            == target_domain
        )
    ].copy()

    matched = test[
        test.in_decision_matched_test
    ].copy()

    # Strict target-domain exclusion.
    assert target_domain not in set(
        train.domain
    )

    assert target_domain not in set(
        val.domain
    )

    assert set(test.domain) == {
        target_domain
    }

    # ID + family separation.
    assert set(train.id).isdisjoint(
        set(val.id)
    )

    assert set(train.id).isdisjoint(
        set(test.id)
    )

    assert set(val.id).isdisjoint(
        set(test.id)
    )

    assert set(
        train.scenario_family_id
    ).isdisjoint(
        set(val.scenario_family_id)
    )

    assert set(
        train.scenario_family_id
    ).isdisjoint(
        set(test.scenario_family_id)
    )

    assert set(
        val.scenario_family_id
    ).isdisjoint(
        set(test.scenario_family_id)
    )

    # Exact text leakage audit.
    for v in TEXT_VIEWS:
        tr = set(
            train[v].astype(str)
        )

        assert tr.isdisjoint(
            set(val[v].astype(str))
        )

        assert tr.isdisjoint(
            set(test[v].astype(str))
        )

    if SMOKE_RUN:
        train = smoke_sample(
            train,
            SMOKE_TRAIN_N,
            101,
        )

        val = smoke_sample(
            val,
            SMOKE_VAL_N,
            102,
        )

        test = smoke_sample(
            test,
            SMOKE_TEST_N,
            103,
        )

        matched = test[
            test.in_decision_matched_test
        ].copy()

    return {
        "target":target_domain,
        "sources":source_domains,
        "train":train.reset_index(drop=True),
        "val":val.reset_index(drop=True),
        "test":test.reset_index(drop=True),
        "matched":matched.reset_index(drop=True),
    }

LODO_SPLITS = {
    d:build_lodo_split(d)
    for d in DOMAINS
}

for target,fold in LODO_SPLITS.items():
    print(
        f"{target:10s} | "
        f"sources={fold['sources']} | "
        f"train={len(fold['train'])} | "
        f"val={len(fold['val'])} | "
        f"test={len(fold['test'])} | "
        f"matched={len(fold['matched'])}"
    )

print("LODO leakage audit: PASS")

In [ ]:
# ============================================================
# CELL 17 — STAGE B: FULL 3-FOLD LODO EXPERIMENT
# ============================================================
LODO_PER_SEED = pd.DataFrame()
LODO_AGG = pd.DataFrame()
LODO_PER_VIEW = pd.DataFrame()

if RUN_FULL_LODO:
    print("\n" + "="*78)
    print("STAGE B — FULL LODO CROSS-DOMAIN EXPERIMENT")
    print("="*78)

    lodo_seeds = (
        [42]
        if SMOKE_RUN
        else LODO_SEEDS
    )

    summary_rows = []
    view_rows = []
    history_rows = []

    for target_domain in DOMAINS:
        fold = LODO_SPLITS[
            target_domain
        ]

        tag = (
            "lodo_excluding_"
            + target_domain.replace(
                "-",
                "_",
            )
        )

        fold_dir = (
            LODO_DIR
            / target_domain.replace(
                "-",
                "_",
            )
        )

        fold_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        print("\n" + "-"*72)
        print(
            "LODO TARGET:",
            target_domain
        )
        print(
            "SOURCE DOMAINS:",
            fold["sources"]
        )
        print("-"*72)

        # CRITICAL:
        # adaptation sees source-domain train only.
        fold_backbone = prepare_tapt_backbone(
            fold["train"],
            tag,
        )

        fold_note_map, fold_rat_dim = (
            build_note_embeddings(
                fold["train"],
                tag,
            )
        )

        for seed in lodo_seeds:
            print(
                f"\n  LODO {target_domain} | seed {seed}"
            )

            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

            model,hist,best_val,state = train_seed(
                fold["train"],
                fold["val"],
                fold_backbone,
                fold_note_map,
                fold_rat_dim,
                seed,
                tag,
            )

            hist["target_domain"] = target_domain
            history_rows.append(hist)

            hist.to_csv(
                fold_dir/f"history_seed{seed}.csv",
                index=False,
            )

            # Source-val temperature:
            # no target-domain labels are used for calibration.
            source_val_pred = predict(
                model,
                fold["val"],
            )

            T_LODO = fit_temperature(
                source_val_pred
            )

            # Unseen target uses GENERIC policy embedding.
            target_pred = apply_temperature(
                predict(
                    model,
                    fold["test"],
                    domain_id_override=len(DOMAINS),
                ),
                T_LODO,
            )

            target_pred.to_csv(
                fold_dir/f"target_predictions_seed{seed}.csv",
                index=False,
            )

            tm = metric_dict(
                target_pred.y_true.values,
                target_pred[
                    [
                        f"p_{x}"
                        for x in LABELS
                    ]
                ].values,
                target_pred.hard_negative.values,
            )

            ci = family_bootstrap(
                target_pred,
                N_BOOT,
                seed,
            )

            matched_macro = np.nan

            if len(fold["matched"]):
                matched_pred = apply_temperature(
                    predict(
                        model,
                        fold["matched"],
                        domain_id_override=len(DOMAINS),
                    ),
                    T_LODO,
                )

                matched_pred.to_csv(
                    fold_dir/f"matched_predictions_seed{seed}.csv",
                    index=False,
                )

                matched_metric = metric_dict(
                    matched_pred.y_true.values,
                    matched_pred[
                        [
                            f"p_{x}"
                            for x in LABELS
                        ]
                    ].values,
                    matched_pred.hard_negative.values,
                )

                matched_macro = (
                    matched_metric["macro_f1"]
                )

            view_table = domain_view_metrics(
                target_pred
            )

            view_table["target_domain"] = (
                target_domain
            )
            view_table["seed"] = seed

            view_rows.append(
                view_table
            )

            summary_rows.append({
                "target_domain":target_domain,
                "source_domains":" + ".join(
                    fold["sources"]
                ),
                "seed":seed,
                "n_target_scenarios":len(
                    fold["test"]
                ),
                "n_target_prompts":len(
                    target_pred
                ),
                "best_source_val_macro_f1":best_val,
                "temperature_source_val":T_LODO,
                "target_macro_f1":tm["macro_f1"],
                "target_f1_ALLOW":tm["f1_ALLOW"],
                "target_f1_REVIEW":tm["f1_REVIEW"],
                "target_f1_BLOCK":tm["f1_BLOCK"],
                "target_binary_macro_f1":tm["binary_macro_f1"],
                "target_threat_recall":tm["threat_recall"],
                "target_threat_auroc":tm["threat_auroc"],
                "target_threat_pr_auc":tm["threat_pr_auc"],
                "target_over_refusal":tm[
                    "hard_negative_over_refusal"
                ],
                "target_ece":tm["ece"],
                "decision_matched_macro_f1":matched_macro,
                "bootstrap_f1_lo":ci[0],
                "bootstrap_f1_median":ci[1],
                "bootstrap_f1_hi":ci[2],
                "generic_policy_at_target":True,
                "peak_gpu_mem_gb":(
                    float(
                        torch.cuda.max_memory_allocated()
                        / 1024**3
                    )
                    if torch.cuda.is_available()
                    else 0.0
                ),
            })

            print(
                f"    target F1={tm['macro_f1']:.4f} | "
                f"REVIEW F1={tm['f1_REVIEW']:.4f} | "
                f"matched={matched_macro:.4f} | "
                f"OR={tm['hard_negative_over_refusal']:.4f}"
            )

            del model, state
            del source_val_pred, target_pred
            if "matched_pred" in locals():
                del matched_pred
            free_memory()

        del fold_note_map
        free_memory()

    LODO_PER_SEED = pd.DataFrame(
        summary_rows
    )

    LODO_PER_SEED.to_csv(
        LODO_DIR/"lodo_summary_by_seed.csv",
        index=False,
    )

    if history_rows:
        pd.concat(
            history_rows,
            ignore_index=True,
        ).to_csv(
            LODO_DIR/"lodo_training_history.csv",
            index=False,
        )

    if view_rows:
        LODO_PER_VIEW = pd.concat(
            view_rows,
            ignore_index=True,
        )

        LODO_PER_VIEW.to_csv(
            LODO_DIR/"lodo_domain_by_view_by_seed.csv",
            index=False,
        )

    LODO_AGG = (
        LODO_PER_SEED
        .groupby(
            "target_domain",
            as_index=False,
        )
        .agg(
            macro_f1_mean=(
                "target_macro_f1",
                "mean",
            ),
            macro_f1_std=(
                "target_macro_f1",
                "std",
            ),
            allow_f1_mean=(
                "target_f1_ALLOW",
                "mean",
            ),
            review_f1_mean=(
                "target_f1_REVIEW",
                "mean",
            ),
            block_f1_mean=(
                "target_f1_BLOCK",
                "mean",
            ),
            matched_f1_mean=(
                "decision_matched_macro_f1",
                "mean",
            ),
            matched_f1_std=(
                "decision_matched_macro_f1",
                "std",
            ),
            threat_recall_mean=(
                "target_threat_recall",
                "mean",
            ),
            threat_auroc_mean=(
                "target_threat_auroc",
                "mean",
            ),
            over_refusal_mean=(
                "target_over_refusal",
                "mean",
            ),
            ece_mean=(
                "target_ece",
                "mean",
            ),
            n_seeds=(
                "seed",
                "nunique",
            ),
        )
    )

    LODO_AGG.to_csv(
        LODO_DIR/"lodo_aggregated_by_target_domain.csv",
        index=False,
    )

    print("\nLODO AGGREGATED RESULTS")
    display(LODO_AGG)

    print(
        "LODO three-domain macro-average:",
        float(
            LODO_AGG.macro_f1_mean.mean()
        )
    )

else:
    print("LODO stage disabled.")

In [ ]:
# ============================================================
# CELL 18 — COMBINED DOMAIN-WISE GENERALIZATION ANALYSIS
# ============================================================
DOMAIN_GENERALIZATION = pd.DataFrame()
LODO_VIEW_AGG = pd.DataFrame()

if (
    len(MAIN_DOMAIN_FINAL)
    and len(LODO_AGG)
):
    main_dom = MAIN_DOMAIN_FINAL[
        [
            "domain",
            "macro_f1",
            "f1_ALLOW",
            "f1_REVIEW",
            "f1_BLOCK",
            "threat_recall",
            "hard_negative_over_refusal",
            "ece",
        ]
    ].copy()

    main_dom = main_dom.rename(
        columns={
            "macro_f1":"in_domain_macro_f1",
            "f1_ALLOW":"in_domain_f1_ALLOW",
            "f1_REVIEW":"in_domain_f1_REVIEW",
            "f1_BLOCK":"in_domain_f1_BLOCK",
            "threat_recall":"in_domain_threat_recall",
            "hard_negative_over_refusal":"in_domain_over_refusal",
            "ece":"in_domain_ece",
        }
    )

    lodo_dom = LODO_AGG.rename(
        columns={
            "target_domain":"domain",
            "macro_f1_mean":"lodo_macro_f1",
            "macro_f1_std":"lodo_macro_f1_std",
            "allow_f1_mean":"lodo_f1_ALLOW",
            "review_f1_mean":"lodo_f1_REVIEW",
            "block_f1_mean":"lodo_f1_BLOCK",
            "matched_f1_mean":"lodo_decision_matched_f1",
            "threat_recall_mean":"lodo_threat_recall",
            "over_refusal_mean":"lodo_over_refusal",
            "ece_mean":"lodo_ece",
        }
    )

    DOMAIN_GENERALIZATION = main_dom.merge(
        lodo_dom,
        on="domain",
        how="inner",
    )

    DOMAIN_GENERALIZATION[
        "macro_f1_generalization_gap"
    ] = (
        DOMAIN_GENERALIZATION[
            "in_domain_macro_f1"
        ]
        - DOMAIN_GENERALIZATION[
            "lodo_macro_f1"
        ]
    )

    DOMAIN_GENERALIZATION[
        "review_f1_generalization_gap"
    ] = (
        DOMAIN_GENERALIZATION[
            "in_domain_f1_REVIEW"
        ]
        - DOMAIN_GENERALIZATION[
            "lodo_f1_REVIEW"
        ]
    )

    DOMAIN_GENERALIZATION[
        "over_refusal_change_lodo_minus_in_domain"
    ] = (
        DOMAIN_GENERALIZATION[
            "lodo_over_refusal"
        ]
        - DOMAIN_GENERALIZATION[
            "in_domain_over_refusal"
        ]
    )

    DOMAIN_GENERALIZATION.to_csv(
        OUT_ROOT/"domain_generalization_comparison.csv",
        index=False,
    )

    print("\nIN-DOMAIN vs LODO GENERALIZATION")
    display(
        DOMAIN_GENERALIZATION[
            [
                "domain",
                "in_domain_macro_f1",
                "lodo_macro_f1",
                "lodo_macro_f1_std",
                "macro_f1_generalization_gap",
                "in_domain_f1_REVIEW",
                "lodo_f1_REVIEW",
                "review_f1_generalization_gap",
                "in_domain_over_refusal",
                "lodo_over_refusal",
            ]
        ]
    )

if len(LODO_PER_VIEW):
    LODO_VIEW_AGG = (
        LODO_PER_VIEW
        .groupby(
            [
                "target_domain",
                "text_view",
            ],
            as_index=False,
        )
        .agg(
            macro_f1_mean=(
                "macro_f1",
                "mean",
            ),
            macro_f1_std=(
                "macro_f1",
                "std",
            ),
            review_f1_mean=(
                "f1_REVIEW",
                "mean",
            ),
            threat_recall_mean=(
                "threat_recall",
                "mean",
            ),
            over_refusal_mean=(
                "hard_negative_over_refusal",
                "mean",
            ),
        )
    )

    LODO_VIEW_AGG.to_csv(
        OUT_ROOT/"lodo_target_domain_by_language_form.csv",
        index=False,
    )

    print("\nLODO DOMAIN × LANGUAGE FORM")
    display(LODO_VIEW_AGG)

In [ ]:
# ============================================================
# CELL 19 — FINAL REPORT PACKAGE + DOWNLOAD
# ============================================================
combined_report = {
    "model":"BanglaGuard-Base",
    "backbone":BACKBONE,
    "main_experiment_run":bool(
        RUN_MAIN_EXPERIMENT
    ),
    "lodo_experiment_run":bool(
        RUN_FULL_LODO
    ),
    "main_seeds":(
        [42]
        if SMOKE_RUN
        else MAIN_SEEDS
    ),
    "lodo_seeds":(
        [42]
        if SMOKE_RUN
        else LODO_SEEDS
    ),
    "tapt_tlm":USE_TAPT_TLM,
    "rationale_alignment":USE_RATIONALE_ALIGNMENT,
    "kd":False,
    "strict_lodo":{
        "target_excluded_from_supervised_train":True,
        "target_excluded_from_validation":True,
        "target_excluded_from_tapt_tlm":True,
        "target_excluded_from_rationale_alignment":True,
        "generic_policy_embedding_at_target":True,
    },
    "main_report":MAIN_REPORT,
}

if len(LODO_AGG):
    combined_report[
        "lodo_three_domain_macro_average"
    ] = float(
        LODO_AGG.macro_f1_mean.mean()
    )

with open(
    OUT_ROOT/"combined_run_report.json",
    "w",
) as f:
    json.dump(
        combined_report,
        f,
        indent=2,
    )

# Small report package:
# preserve directory structure and exclude checkpoints/caches.
report_root = (
    OUT_ROOT
    / "report_package"
)

report_root.mkdir(
    parents=True,
    exist_ok=True,
)

for p in OUT_ROOT.rglob("*"):
    if not p.is_file():
        continue

    if report_root in p.parents:
        continue

    if p.suffix.lower() not in {
        ".csv",
        ".json",
        ".txt",
    }:
        continue

    rel = p.relative_to(
        OUT_ROOT
    )

    dst = (
        report_root
        / rel
    )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        p,
        dst,
    )

zip_out = (
    WORK_ROOT
    / "BanglaGuard_Base_FULL_Benchmark_PLUS_LODO_results.zip"
)

shutil.make_archive(
    str(zip_out).replace(
        ".zip",
        "",
    ),
    "zip",
    report_root,
)

print("\n" + "="*78)
print("FULL BANGLAGUARD-BASE + LODO PIPELINE COMPLETE")
print("="*78)
print("Output root:", OUT_ROOT)
print("Result ZIP :", zip_out)

if len(DOMAIN_GENERALIZATION):
    print("\nFinal domain-generalization table:")
    display(
        DOMAIN_GENERALIZATION
    )

print("\nDownloading report ZIP...")
colab_files.download(
    str(zip_out)
)